# P08 — Konvolucija zvuka s impulsnim odzivom prostorije

Bilježnica uz predavanje P08 i **poglavlje 8 udžbenika *Signali i sustavi*** (Konvolucija i
impulsni odziv). Suhi zvučni zapis, snimljen bez odjeka, konvoluiramo sa snimkom pucnja u
prostoriji i čujemo kako bi isti zvuk zvučao u toj prostoriji.

Ćelije pokrenite redom (Shift + Enter) ili izbornikom *Run → Run All Cells*. Za zvuk trebaju
zvučnici ili slušalice. Bilježnica radi bez interneta.

## 1. Konvolucija i impulsni odziv

**Impulsni odziv** $h(t)$ linearnog vremenski nepromjenjivog (LTI) sustava je njegov odziv na
Diracovu deltu $\delta(t)$. Odziv na bilo koji ulaz $x(t)$ je **konvolucija** ulaza s
impulsnim odzivom:

$$y(t) = (x*h)(t) = \int_{-\infty}^{\infty} x(\tau)\,h(t-\tau)\,d\tau .$$

Računalo radi s uzorcima $x[n]=x(nT)$, gdje je $T=1/f_s$ razmak uzoraka. Integral tada
postaje zbroj, **konvolucijska suma** (poglavlje 12), do faktora $T$ koji ionako uklanja
normiranje glasnoće:

$$y[n] = \sum_{k} x[k]\,h[n-k] .$$

**Prostorija kao sustav.** Dok izvor zvuka i mikrofon stoje na istim mjestima, put zvuka
kroz zrak i odbijanje od zidova približno su linearni i vremenski nepromjenjivi. Pucanj (ili
prasak balona) je vrlo kratak i jak zvuk, pa je približno delta. Njegova snimka je zato
**impulsni odziv prostorije**: izravni zvuk, rane refleksije od zidova i odjek koji polako
trne. Svaki uzorak suhog zapisa $x[k]$ pokrene u prostoriji kopiju odziva, umanjenu za
$x[k]$ i pomaknutu za $k$ uzoraka. Zbroj tih kopija je zvuk koji čujemo.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.io import wavfile
from math import gcd
from scipy.signal import fftconvolve, resample_poly
from IPython.display import Audio, display
import ipywidgets as widgets

ZVUK = Path("zvuk")
BOJA_X, BOJA_H, BOJA_Y = "#840B55", "#5B6770", "#222222"     # bordo, siva, tamna


def broj(v, decimala=2):
    """Broj s decimalnim zarezom (1,50)."""
    return f"{v:.{decimala}f}".replace(".", ",")


zarez = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))   # decimalni zarez na osima
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})

## 2. Učitavanje zapisa

Suhi zapis $x[n]$ snimljen je bez odjeka, a impulsni odziv $h[n]$ je snimka pucnja u
prostoriji. U ovoj inačici oba su zapisa **zamjenska**: generira ih skripta
`alati/zamjenski_zvuk.py`: suhi zapis ima dva kratka klika, četiri tona i završni akord, a
impulsni odziv izravni zvuk, rane refleksije i odjek od oko 1 s. Izvor i licenca svakog
zapisa navedeni su u `LICENSE-sadrzaj.md`.

Vlastitu snimku (WAV) stavite u mapu `moj_zvuk/` (nije u gitu, pa `zvuk/` ostaje
nepromijenjen) i u ćeliji ispod učitajte je s
`ucitaj("ime.wav", fs=fs, mapa="moj_zvuk")`. Funkcija
`ucitaj` pretvara stereo zapis u mono, prihvaća 8-, 16- i 32-bitne zapise te zapis s drugom
frekvencijom uzorkovanja (npr. 48 kHz s mobitela) preuzorkuje na frekvenciju suhog zapisa.

In [ ]:
def ucitaj(ime, fs=None, mapa=ZVUK):
    """WAV iz mape -> (frekvencija uzorkovanja, mono signal u rasponu −1 … 1).

    Zadani ``fs``: zapis s drugom frekvencijom uzorkovanja preuzorkuje se na ``fs``."""
    fs_zapisa, s = wavfile.read(Path(mapa) / ime)
    if s.dtype == np.uint8:                # 8-bitni WAV: 0 … 255, tišina = 128
        s = (s.astype(float) - 128) / 128
    elif np.issubdtype(s.dtype, np.integer):
        s = s / np.iinfo(s.dtype).max
    s = np.asarray(s, dtype=float)
    if s.ndim == 2:                        # stereo -> mono
        s = s.mean(axis=1)
    if fs is not None and fs_zapisa != fs:
        d = gcd(fs, fs_zapisa)
        s = resample_poly(s, fs // d, fs_zapisa // d)
        print(f"{ime}: preuzorkovano s {fs_zapisa} Hz na {fs} Hz")
        fs_zapisa = fs
    return fs_zapisa, s


fs, x = ucitaj("suhi_zvuk.wav")
_, h = ucitaj("odziv_pucanj.wav", fs=fs)
Tx, Th = len(x) / fs, len(h) / fs
print(f"frekvencija uzorkovanja f_s = {fs} Hz")
print(f"suhi zapis x:       N_x = {len(x)} uzoraka, T_x = {broj(Tx)} s")
print(f"impulsni odziv h:   N_h = {len(h)} uzoraka, T_h = {broj(Th)} s")

## 3. Slušanje: suhi zapis i pucanj u prostoriji

Najprije poslušajte oba zapisa. Suhi zapis zvuči „mrtvo”, kao u studiju. Impulsni odziv
počinje oštrim praskom (izravni zvuk), a zatim se čuje odjek koji trne.

In [ ]:
print("Suhi zapis x[n]:")
display(Audio(x, rate=fs))
print("Impulsni odziv h[n] (pucanj u prostoriji):")
display(Audio(h, rate=fs))

## 4. Konvolucija $y = x * h$

Izravni zbroj iz formule trebao bi oko $N_x \cdot N_h \approx 9\cdot 10^9$ množenja.
Funkcija `scipy.signal.fftconvolve` daje isti rezultat preko brze Fourierove transformacije
(FFT, poglavlja 9 i 13) u djeliću sekunde. Rezultat normiramo na vršnu vrijednost 0,9, da
zvuk ne bude preglasan ni izobličen.

In [ ]:
y = fftconvolve(x, h)
assert len(y) == len(x) + len(h) - 1
y = 0.9 * y / np.max(np.abs(y))            # normiranje glasnoće
Ty = len(y) / fs
print(f"y: N_y = N_x + N_h − 1 = {len(x)} + {len(h)} − 1 = {len(y)} uzoraka")
print(f"trajanje: T_y = {broj(Ty)} s ≈ T_x + T_h = {broj(Tx)} s + {broj(Th)} s")
print("Rezultat y[n], suhi zapis u prostoriji:")
display(Audio(y, rate=fs))

## 5. Tri signala na istoj vremenskoj osi

Sva tri signala crtamo na istoj vremenskoj osi. Izlaz $y$ traje dulje od ulaza $x$, i to
upravo za trajanje impulsnog odziva $T_h$: zadnji uzorak ulaza pokrene još jednu cijelu kopiju
odziva. Iza kraja ulaza (crtkana crta) završni akord u izlazu ne prestaje naglo, nego trne.

In [ ]:
fig, osi = plt.subplots(3, 1, figsize=(10, 6.5), sharex=True)
signali = [(x, BOJA_X, "$x[n]$", "suhi zapis $x[n]$"),
           (h, BOJA_H, "$h[n]$", "impulsni odziv $h[n]$ (pucanj u prostoriji)"),
           (y, BOJA_Y, "$y[n]$", "izlaz $y[n] = (x*h)[n]$")]
for ax, (s, boja, os_y, naslov) in zip(osi, signali):
    ax.plot(np.arange(len(s)) / fs, s, color=boja, lw=0.6)
    ax.set_ylabel(os_y)
    ax.set_title(naslov, loc="left", fontsize=10)
    ax.set_ylim(-1.05, 1.05)
    for granica, crta in ((Tx, "--"), (Ty, ":")):
        ax.axvline(granica, color="k", ls=crta, lw=1)
    ax.yaxis.set_major_formatter(zarez)
osi[2].annotate("", xy=(Ty, -0.8), xytext=(Tx, -0.8),
                arrowprops=dict(arrowstyle="<->", color="k"))
osi[2].text((Tx + Ty) / 2, -0.75, f"produljenje ≈ $T_h$ = {broj(Th)} s",
            ha="center", va="bottom")
osi[2].set_xlabel("vrijeme $t$ / s   (crtkano: kraj ulaza $T_x$; točkasto: kraj izlaza $T_y$)")
osi[2].set_xlim(0, Ty * 1.02)
osi[2].xaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

Na linearnom prikazu tihi kraj odjeka gotovo se ne vidi. Zato crtamo i **razinu u
decibelima**: srednju snagu u kliznom prozoru od 20 ms. Klizni prosjek je i sam konvolucija,
s pravokutnim impulsom (pomični prosjek s predavanja). Razina izlaza nakon kraja ulaza pada
gotovo pravocrtno, za 60 dB u vremenu odjeka $RT_{60} \approx 1$ s, i odjek traje sve do kraja
izlaza.

In [ ]:
def razina_db(s, prozor=0.02):
    """Razina u dB: srednja snaga u kliznom prozoru (s), 0 dB = najveća vrijednost."""
    n = int(prozor * fs)
    snaga = np.maximum(fftconvolve(s**2, np.ones(n) / n, mode="same"), 0)
    return 10 * np.log10(snaga / snaga.max() + 1e-12)


fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(np.arange(len(x)) / fs, razina_db(x), color=BOJA_X, lw=1, label="ulaz $x$")
ax.plot(np.arange(len(y)) / fs, razina_db(y), color=BOJA_Y, lw=1, ls="--", label="izlaz $y$")
ax.axvline(Tx, color="k", ls="--", lw=1)
ax.axvline(Ty, color="k", ls=":", lw=1)
ax.axhline(-60, color=BOJA_H, lw=0.8)
ax.text(Ty - 0.05, -57, "−60 dB", ha="right", color=BOJA_H)
ax.set_ylim(-80, 3)
ax.set_xlim(0, Ty * 1.02)
ax.set_xlabel("vrijeme $t$ / s")
ax.set_ylabel("razina / dB")
ax.set_title("Razina u decibelima: odjek traje i nakon kraja ulaza", loc="left", fontsize=10)
ax.legend(loc="upper right")
ax.xaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

Pogledajmo pobliže početak impulsnog odziva. Prvi impuls je izravni zvuk, a nekoliko
idućih, unutar nekoliko desetaka milisekundi, su rane refleksije od zidova, poda i stropa.
Nakon njih refleksija je toliko da se stapaju u šum koji eksponencijalno trne: to je odjek.

In [ ]:
n = int(0.06 * fs)                          # prvih 60 ms
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(1000 * np.arange(n) / fs, h[:n], color=BOJA_H, lw=0.8)
ax.set_xlabel("vrijeme $t$ / ms")
ax.set_ylabel("$h[n]$")
ax.set_title("Početak odziva: izravni zvuk ($t = 0$), rane refleksije, početak odjeka",
              loc="left", fontsize=10)
ax.yaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

## 6. Klizači: udio odjeka i duljina odziva

Klizač **udio odjeka** $a$ miješa suhi i „mokri” zvuk:
$y_a = (1-a)\,x + a\,(x * h_T)$. Za $a = 0$ čuje se samo suhi zapis, a za $a = 1$ samo
zvuk u prostoriji. Klizač **odziv / s** skraćuje impulsni odziv na prvih $T$ sekundi
($h_T$): kraći odziv znači kraći odjek, kao u manjoj ili prigušenijoj prostoriji, a izlaz je
dulji od ulaza samo za $T$. Nakon pomaka klizača bilježnica ponovno računa konvoluciju i nudi
novi zvuk za puštanje.

Ako se klizači ne prikažu, pozovite funkciju izravno, npr. `odjek(udio=0.5, odziv_s=0.3)`.

In [ ]:
def odjek(udio=0.7, odziv_s=None):
    """Mješavina suhog i mokrog zvuka; odziv skraćen na odziv_s sekundi (None: cijeli)."""
    n_h = len(h) if odziv_s is None else min(len(h), max(2, int(odziv_s * fs)))
    h_T = h[:n_h].copy()
    n_rub = min(n_h, int(0.01 * fs))            # 10 ms mekog kraja: bez klika na rezu
    h_T[n_h - n_rub:] *= np.linspace(1, 0, n_rub)
    mokri = fftconvolve(x, h_T)
    mokri /= np.max(np.abs(mokri))
    suhi = np.pad(x / np.max(np.abs(x)), (0, len(mokri) - len(x)))
    y_a = (1 - udio) * suhi + udio * mokri
    y_a = 0.9 * y_a / np.max(np.abs(y_a))
    fig, ax = plt.subplots(figsize=(10, 2.5))
    ax.plot(np.arange(len(y_a)) / fs, y_a, color=BOJA_Y, lw=0.6)
    ax.axvline(Tx, color="k", ls="--", lw=1)
    ax.set_xlim(0, Tx + Th)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel("vrijeme $t$ / s")
    ax.set_title(f"udio odjeka {broj(udio)}, odziv {broj(n_h / fs)} s: "
                  f"izlaz traje {broj(len(y_a) / fs)} s", loc="left", fontsize=10)
    ax.xaxis.set_major_formatter(zarez)
    ax.yaxis.set_major_formatter(zarez)
    fig.tight_layout()
    plt.show()
    display(Audio(y_a, rate=fs))


klizaci = widgets.interactive(
    odjek,
    udio=widgets.FloatSlider(value=0.7, min=0, max=1, step=0.05, description="udio odjeka",
                             continuous_update=False),
    odziv_s=widgets.FloatSlider(value=round(Th, 2), min=0.05, max=round(Th, 2), step=0.05,
                                description="odziv / s", continuous_update=False),
)
display(klizaci)

## 7. Veza s formulom $y = x * h$

- **Zbroj pomaknutih kopija.** Svaki uzorak $x[k]$ pokrene kopiju odziva $x[k]\,h[n-k]$, a
  izlaz je njihov zbroj: to je upravo formula $y[n]=\sum_k x[k]\,h[n-k]$. Ćelija ispod
  provjerava da `fftconvolve` daje isto što i zbroj iz formule.
- **Klik je gotovo delta.** Nakon klika čuje se sam odjek prostorije, jer je
  $\delta * h = h$. Delta je jedinica konvolucije: $x * \delta = x$.
- **Produljenje.** $N_y = N_x + N_h - 1$, odnosno $T_y \approx T_x + T_h$, kao u grafičkoj
  konvoluciji dvaju pravokutnih impulsa (trajanja 1 s i 2 s daju rezultat od 3 s).
- **Jeka s predavanja.** Odziv $h(t)=\delta(t)+\tfrac{1}{2}\,\delta(t-2)$ ima dva impulsa
  i daje jednu jeku. Prostorija ima tisuće refleksija, koje se stapaju u odjek.
- **Razmazani tonovi.** Ton još zvuči dok počinje idući, pa se tonovi preklapaju. Zato se
  govor u velikoj dvorani s dugim odjekom teže razumije.

Za čitanje: udžbenik *Signali i sustavi*, **poglavlje 8, Konvolucija i impulsni odziv**
(grafička konvolucija, svojstva konvolucije i primjer jeke), i predavanje P08.

In [ ]:
# Provjera 1: fftconvolve daje isto što i zbroj iz formule (kratki isječci signala).
# Isječak x počinje na prvom kliku: suhi zapis do njega šuti, a s nulama provjera ne bi
# ništa dokazala.
p = int(np.flatnonzero(np.abs(x) > 0.01)[0])
xk, hk = x[p:p + 300], h[:200]
assert np.max(np.abs(xk)) > 0.1 and np.max(np.abs(hk)) > 0.1, "isječak šuti"
y_formula = np.array([sum(xk[k] * hk[n - k] for k in range(len(xk)) if 0 <= n - k < len(hk))
                      for n in range(len(xk) + len(hk) - 1)])
razlika = np.max(np.abs(fftconvolve(xk, hk) - y_formula))
print(f"najveća razlika fftconvolve − zbroj iz formule: {razlika:.1e}")
assert razlika < 1e-9

# Provjera 2: delta je jedinica konvolucije, a pomaknuta delta kasni signal
delta = np.zeros(fs // 2)
delta[0] = 1
assert np.allclose(fftconvolve(x, delta)[:len(x)], x)
kasnjenje = fs // 4                          # 0,25 s
delta_T = np.zeros(fs // 2)
delta_T[kasnjenje] = 1
assert np.allclose(fftconvolve(x, delta_T)[kasnjenje:kasnjenje + len(x)], x)
print("x * δ = x i x * δ(t − 0,25 s) = x(t − 0,25 s): provjereno")

## Pokušajte sami

1. Postavite udio odjeka na 1 i skraćujte odziv od 1,5 s prema 0,05 s. Kada odjek prestaje
   biti čujan, a kada se još čuju samo rane refleksije?
2. Napravite odziv s drugim vremenom odjeka u zasebnoj mapi (zvuk/ ostaje nepromijenjen):
   `python alati/zamjenski_zvuk.py moj_zvuk --rt60 2.5` (ili `--rt60 0.3`). U odjeljku 2
   učitajte ga s `_, h = ucitaj("odziv_pucanj.wav", fs=fs, mapa="moj_zvuk")` i ponovno
   pokrenite bilježnicu od te ćelije nadalje. Odziv traje 1,5·RT60, najmanje 1,5 s, pa klizač
   duljine prati.
3. Snimite pljesak ili prasak balona u hodniku ili stubištu, spremite ga kao WAV u
   `moj_zvuk/` (nije u gitu) i upotrijebite ga kao impulsni odziv u odjeljku 2:
   `_, h = ucitaj("pljesak.wav", fs=fs, mapa="moj_zvuk")`.
4. Izračunajte i poslušajte $h * h$. Koju prostoriju (ili spoj dviju prostorija) opisuje?
   Uputa: asocijativnost konvolucije, $(x * h) * h = x * (h * h)$.

---
Zvučni zapisi: vlastiti zamjenski zapisi, izvor i licenca u `LICENSE-sadrzaj.md`. Kod je pod
licencom MIT (`LICENSE`), a tekst pod CC BY 4.0.